In [ ]:
# ============================================================
# 05_counterfactual_dice.ipynb
# DiCE counterfactual explanations for the non-diabetic high-risk
# cohort -- individual-level algorithmic recourse (interpretability
# / compliance control).
#
# Input : data/processed/final_pipeline_{sex}.pkl,
#         data/processed/scored_{sex}.csv,
#         data/processed/highrisk_nondiabetic_{sex}.csv
# Output: data/processed/counterfactuals_{sex}.csv,
#         results/tables/counterfactual_summary.csv,
#         results/figures/counterfactual_scatter.(png|pdf),
#         results/figures/counterfactual_reduction.(png|pdf)
#
# Actionability tiers: age immutable; anthropometric/diet/lifestyle
# features actionable. Target class = non-diabetic (0). Up to 4
# diverse counterfactuals per individual; the best (largest risk
# reduction) is retained.
#
# Set env MAX_CASES to cap individuals per sex for a quick run
# (default: all). Figures grayscale, dpi 600, PNG + PDF, no captions.
# ============================================================

import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import dice_ml

SEED = 42
np.random.seed(SEED)
MAX_CASES = os.environ.get("MAX_CASES")
MAX_CASES = int(MAX_CASES) if MAX_CASES else None
TOTAL_CFS = 4

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

GRAY = ["#000000", "#595959", "#8C8C8C", "#BFBFBF", "#D9D9D9"]
sns.set_theme(style="whitegrid", context="paper")
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 11, "axes.edgecolor": "black",
    "axes.linewidth": 0.8, "axes.grid": True, "grid.color": "0.88", "grid.linewidth": 0.6,
})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

cat_col = ["HE_obe","BO1_1","BO1_2","BO1_3","BD1_11","BD2_1","BS3_1","BE3_71",
           "BE3_75","BE3_81","BE3_91","pa_aerobic","L_BR_FQ","BP1","mh_stress",
           "incm","ho_incm","edu","BH1"]
num_col = ["HE_BMI","HE_wc","HE_wt","N_SUGAR","N_CHO","N_EN","age"]
feature_col = cat_col + num_col
TARGET = "HE_DM_HbA1c"
# Immutable features (not varied by DiCE)
IMMUTABLE = ["age"]
vary = [c for c in feature_col if c not in IMMUTABLE]

all_cf, summary_rows = [], []
payloads = {"male": [], "female": []}  # rich payloads for LLM guide generation (nb 08)
for sex in ["male", "female"]:
    bundle = joblib.load(PROC / f"final_pipeline_{sex}.pkl")
    pipe = bundle["pipe"]
    train = pd.read_csv(PROC / f"scored_{sex}.csv")[feature_col + [TARGET]]
    hr = pd.read_csv(PROC / f"highrisk_nondiabetic_{sex}.csv")[feature_col].reset_index(drop=True)
    if MAX_CASES:
        hr = hr.iloc[:MAX_CASES].copy()

    # DiCE data + model (sklearn backend, raw-feature pipeline)
    dd = dice_ml.Data(dataframe=train, continuous_features=num_col, outcome_name=TARGET)
    dm = dice_ml.Model(model=pipe, backend="sklearn")
    exp = dice_ml.Dice(dd, dm, method="genetic")

    # permitted ranges: allow continuous actionable features down to dataset min
    pr = {}
    for c in num_col:
        if c in IMMUTABLE:
            continue
        pr[c] = [float(train[c].min()), float(train[c].max())]

    orig_prob = pipe.predict_proba(hr)[:, 1]
    n_ok = 0
    for i in range(len(hr)):
        row = hr.iloc[[i]]
        try:
            cf = exp.generate_counterfactuals(
                row, total_CFs=TOTAL_CFS, desired_class=0,
                features_to_vary=vary, permitted_range=pr, verbose=False)
            cfdf = cf.cf_examples_list[0].final_cfs_df
            if cfdf is None or len(cfdf) == 0:
                continue
            cfX = cfdf[feature_col]
            cf_probs = pipe.predict_proba(cfX)[:, 1]
            j = int(np.argmin(cf_probs))              # best (lowest-risk) CF
            best_prob = float(cf_probs[j])
            # dominant changed feature (largest standardized change among varied)
            deltas = {}
            for c in num_col:
                if c in IMMUTABLE:
                    continue
                d0 = float(row[c].iloc[0]); d1 = float(cfX.iloc[j][c])
                if abs(d1 - d0) > 1e-9:
                    deltas[c] = d1 - d0
            dom = min(deltas, key=lambda k: deltas[k]) if deltas else ""
            # full set of changed features (continuous + categorical) for this CF
            changed = list(deltas.keys())
            for c in cat_col:
                if float(cfX.iloc[j][c]) != float(row[c].iloc[0]):
                    changed.append(c)
            all_cf.append(dict(sex=sex, idx=i, orig_prob=float(orig_prob[i]),
                               cf_prob=best_prob, risk_reduction=float(orig_prob[i]) - best_prob,
                               dominant_feature=dom,
                               changed_features="|".join(sorted(set(changed))),
                               waist_delta=float(cfX.iloc[j]["HE_wc"]) - float(row["HE_wc"].iloc[0])))
            # rich payload for LLM guide generation (notebook 08)
            cf_detail = []
            for c in sorted(set(changed)):
                o0 = float(row[c].iloc[0]); o1 = float(cfX.iloc[j][c])
                cf_detail.append(dict(feature=c, original_value=o0, cf_value=o1,
                                      delta=(o1 - o0) if c in num_col else None))
            payloads[sex].append(dict(
                gender=sex, age=int(row["age"].iloc[0]),
                original_prob=float(orig_prob[i]), cf_prob=best_prob,
                prob_reduction=float(orig_prob[i]) - best_prob,
                changed_features=cf_detail,
                static_features={"Age (years)": int(row["age"].iloc[0])}))
            n_ok += 1
        except Exception:
            continue
    print(f"{sex:7s}: counterfactuals generated for {n_ok}/{len(hr)} individuals")

    sub = pd.DataFrame([r for r in all_cf if r["sex"] == sex])
    if len(sub):
        summary_rows.append(dict(
            Sex=sex, N=len(sub),
            Mean_orig_prob=round(sub["orig_prob"].mean(), 3),
            Mean_cf_prob=round(sub["cf_prob"].mean(), 3),
            Mean_risk_reduction=round(sub["risk_reduction"].mean(), 3),
            Dominant_feature=sub["dominant_feature"].mode().iat[0] if sub["dominant_feature"].ne("").any() else "",
            Mean_abs_waist_delta=round(sub["waist_delta"].abs().mean(), 2)))

cf_df = pd.DataFrame(all_cf)
if cf_df.empty:
    cf_df = pd.DataFrame(columns=["sex","idx","orig_prob","cf_prob","risk_reduction","dominant_feature","changed_features","waist_delta"])
cf_df.to_csv(PROC / "counterfactuals_combined.csv", index=False)
for sex in ["male", "female"]:
    cf_df[cf_df["sex"] == sex].to_csv(PROC / f"counterfactuals_{sex}.csv", index=False)
summ = pd.DataFrame(summary_rows)
summ.to_csv(TAB / "counterfactual_summary.csv", index=False)
print("\nCounterfactual summary:\n", summ.to_string(index=False))

# ---- Save rich payloads for LLM guide generation (notebook 08) ----
for sex in ["male", "female"]:
    with open(PROC / f"llm_payloads_{sex}.json", "w", encoding="utf-8") as f:
        json.dump(payloads[sex], f, ensure_ascii=False, indent=2)
print("LLM payloads saved: data/processed/llm_payloads_{male,female}.json")

# ---- Figure: original vs counterfactual risk probability ----
fig, ax = plt.subplots(figsize=(4.8, 4.6))
for sex, mk, gc in [("male", "o", GRAY[1]), ("female", "^", GRAY[3])]:
    s = cf_df[cf_df["sex"] == sex]
    ax.scatter(s["orig_prob"], s["cf_prob"], s=18, marker=mk, facecolor=gc,
               edgecolor="black", linewidth=0.3, alpha=0.7, label=sex.capitalize())
ax.plot([0, 1], [0, 1], color=GRAY[0], lw=1.0, linestyle="--")
ax.set_xlabel("Original predicted risk"); ax.set_ylabel("Counterfactual predicted risk")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=9)
save_fig(fig, "counterfactual_scatter")

# ---- Figure: distribution of risk reduction by sex ----
fig, ax = plt.subplots(figsize=(5.2, 3.6))
for sex, gc, ls in [("male", GRAY[1], "-"), ("female", GRAY[3], "--")]:
    s = cf_df[cf_df["sex"] == sex]["risk_reduction"]
    if len(s):
        sns.kdeplot(s, ax=ax, color=gc, fill=True, alpha=0.35, linewidth=1.6, label=sex.capitalize())
        ax.axvline(s.mean(), color=gc, lw=1.2, linestyle=ls)
ax.set_xlabel("Risk reduction (original - counterfactual probability)")
ax.set_ylabel("Density"); ax.legend(frameon=False, fontsize=9)
save_fig(fig, "counterfactual_reduction")
print("Figures written: counterfactual_scatter, counterfactual_reduction (png+pdf).")
